# Graph Construction
**GNN-based Fraud Review Detection | SKK DScover**

---
## 목표
- 서브그래프 CSV → PyTorch Geometric `Data` 객체로 변환
- 노드 피처 구성 (BGE-large BERT 임베딩 1024dim + 수치 피처 7dim)
- 엣지 설계 (기본 Relation 1개 + 커스텀 Relation 1개)

## 엣지 설계 전략
| 구분 | Relation | 설명 |
|------|----------|------|
| **기본 (필수)** | **R-U-R** | 같은 유저가 작성한 리뷰 간 연결 |
| **커스텀 (창의)** | **R-W-R** | 같은 식당에 7일 이내 작성된 리뷰 간 연결 (Time Window) |

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from itertools import combinations
import torch
from torch_geometric.data import Data
import warnings
warnings.filterwarnings('ignore')

DATA_PATH  = '../data/subgraph.csv'
EMBED_PATH = '../data/bert_embeddings.npy'
SAVE_PATH  = '../data/graph.pt'
RANDOM_STATE = 42
TIME_WINDOW_DAYS = 7  # 커스텀 Relation 시간 윈도우

c:\Users\user\anaconda3\envs\gnn\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\user\anaconda3\envs\gnn\Lib\site-packages\torch_scatter\_version_cuda.pyd
  import torch_geometric.typing
c:\Users\user\anaconda3\envs\gnn\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\user\anaconda3\envs\gnn\Lib\site-packages\torch_sparse\_version_cuda.pyd
  import torch_geometric.typing


## 1. 데이터 로드

In [ ]:
df = pd.read_csv(DATA_PATH, index_col=0, parse_dates=['date'])
df = df.reset_index(drop=True)  # 0-based 연속 인덱스 확보

print(f'shape: {df.shape}')
print(f'columns: {df.columns.tolist()}')
print(f'label: {df["label"].value_counts().to_dict()}')
df.head(3)

: 

: 

## 2. 노드 피처 구성

### 피처 목록
| 피처 | 설명 |
|------|------|
| BGE-large BERT (1024dim) | 리뷰 텍스트 → 의미 표현 (MTEB 최상위) |
| rating_norm | 별점 정규화 (1~5) |
| text_len_norm | 텍스트 길이 (정규화) |
| word_count_norm | 단어 수 (정규화) |
| is_extreme | 별점 1점 또는 5점 여부 |
| user_review_cnt | 서브그래프 내 유저의 총 리뷰 수 |
| month_sin / month_cos | 월 주기성 인코딩 |

In [ ]:
# BERT 임베딩 로드 (04_bert_embedding.ipynb 먼저 실행 필요)
assert os.path.exists(EMBED_PATH), f'임베딩 파일 없음: {EMBED_PATH}\n04_bert_embedding.ipynb를 먼저 실행하세요.'
bert_feats = np.load(EMBED_PATH).astype(np.float32)
print(f'BERT 임베딩 로드 완료: {bert_feats.shape}  (nodes x 1024)')

: 

: 

In [ ]:
# 수치 피처
scaler = StandardScaler()

# 텍스트 길이 / 단어 수
df['text_len'] = df['text'].fillna('').apply(len)
df['word_count'] = df['text'].fillna('').apply(lambda x: len(x.split()))

# 유저별 리뷰 수 (서브그래프 내)
user_cnt = df.groupby('user_id').size().rename('user_review_cnt')
df = df.join(user_cnt, on='user_id')

# 월 주기성 인코딩
df['month'] = df['date'].dt.month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# 극단 별점 여부
df['is_extreme'] = df['rating'].apply(lambda x: 1 if x in [1.0, 5.0] else 0)

num_cols = ['rating', 'text_len', 'word_count', 'user_review_cnt',
            'month_sin', 'month_cos', 'is_extreme']
num_feats = scaler.fit_transform(df[num_cols].values)

print(f'수치 피처 shape: {num_feats.shape}')

: 

: 

In [ ]:
# 피처 결합
X = np.concatenate([bert_feats, num_feats], axis=1).astype(np.float32)
print(f'최종 노드 피처 shape: {X.shape}  (BERT 1024 + 수치 7 = {X.shape[1]} dims)')

: 

: 

## 3. 엣지 구성

### 3-1. 기본 Relation — R-U-R (같은 유저 리뷰 간 연결)

In [ ]:
print('R-U-R 엣지 생성 중...')

rur_edges = []
for uid, group in df.groupby('user_id'):
    idxs = group.index.tolist()
    if len(idxs) >= 2:
        for i, j in combinations(idxs, 2):
            rur_edges.append((i, j))
            rur_edges.append((j, i))  # 무방향 → 양방향

rur_edges = np.array(rur_edges, dtype=np.int64)
print(f'R-U-R 엣지 수: {len(rur_edges):,}  (양방향 포함)')

: 

: 

### 3-2. 커스텀 Relation — R-W-R (같은 식당, 7일 이내 리뷰 간 연결)

> **설계 근거**: 조직적 어뷰징은 짧은 기간 내에 같은 식당에 집중적으로 가짜 리뷰를 남기는 패턴을 보임.
> R-T-R(월 단위)보다 좁은 7일 윈도우를 적용해 더 의심스러운 연결만 포착.

In [ ]:
print(f'R-W-R 엣지 생성 중 (같은 식당 + {TIME_WINDOW_DAYS}일 이내)...')

rwr_edges = []
for pid, group in df.groupby('prod_id'):
    group = group.sort_values('date')
    idxs = group.index.tolist()
    dates = group['date'].values
    for k in range(len(idxs)):
        for l in range(k + 1, len(idxs)):
            diff = (dates[l] - dates[k]).astype('timedelta64[D]').astype(int)
            if diff <= TIME_WINDOW_DAYS:
                rwr_edges.append((idxs[k], idxs[l]))
                rwr_edges.append((idxs[l], idxs[k]))
            else:
                break  # 정렬되어 있으므로 이후는 모두 범위 초과

rwr_edges = np.array(rwr_edges, dtype=np.int64) if rwr_edges else np.empty((0, 2), dtype=np.int64)
print(f'R-W-R 엣지 수: {len(rwr_edges):,}  (양방향 포함)')

: 

: 

In [ ]:
# 엣지 통합
all_edges = np.concatenate([rur_edges, rwr_edges], axis=0)

# 중복 제거
all_edges = np.unique(all_edges, axis=0)

edge_index = torch.tensor(all_edges.T, dtype=torch.long)

print(f'\n=== 엣지 요약 ===')
print(f'R-U-R 엣지  : {len(rur_edges):,}')
print(f'R-W-R 엣지  : {len(rwr_edges):,}')
print(f'통합 (중복제거): {edge_index.shape[1]:,}')
print(f'평균 degree : {edge_index.shape[1] / len(df):.2f}')

: 

: 

## 4. PyTorch Geometric Data 객체 생성

In [ ]:
# 마스크 생성
train_mask = torch.tensor(df['split'].values == 'train', dtype=torch.bool)
test_mask  = torch.tensor(df['split'].values == 'test',  dtype=torch.bool)

# 레이블
y = torch.tensor(df['label'].values, dtype=torch.long)

# 노드 피처
x = torch.tensor(X, dtype=torch.float)

# Data 객체
data = Data(
    x=x,
    edge_index=edge_index,
    y=y,
    train_mask=train_mask,
    test_mask=test_mask
)

print(data)
print(f'\n노드 수     : {data.num_nodes:,}')
print(f'엣지 수     : {data.num_edges:,}')
print(f'피처 차원   : {data.num_features}')
print(f'Train 노드  : {train_mask.sum().item():,}')
print(f'Test 노드   : {test_mask.sum().item():,}')
print(f'\n그래프 연결성 확인:')
print(f'  isolated 노드 여부: {not data.has_isolated_nodes()}')

: 

: 

## 5. 그래프 저장

In [ ]:
import os

torch.save(data, SAVE_PATH)
size_mb = os.path.getsize(SAVE_PATH) / (1024 ** 2)

print(f'저장 완료: {SAVE_PATH}')
print(f'파일 크기: {size_mb:.2f} MB')

: 

: 

## 6. 그래프 구성 요약 (보고서 기재용)

In [ ]:
print('=' * 55)
print('           그래프 구성 요약')
print('=' * 55)
print(f'노드 (리뷰 단위)    : {data.num_nodes:,}개')
print(f'엣지 총합           : {data.num_edges:,}개')
print(f'노드 피처 차원      : {data.num_features}dim')
print(f'  - BGE-large BERT  : 1024dim')
print(f'  - 수치 피처        : 7dim (rating, text_len, word_count,')
print(f'                       user_review_cnt, month_sin, month_cos, is_extreme)')
print('-' * 55)
print(f'[기본 Relation] R-U-R')
print(f'  동일 유저 리뷰 간 연결 | {len(rur_edges):,}개')
print(f'[커스텀 Relation] R-W-R (Time Window {TIME_WINDOW_DAYS}일)')
print(f'  같은 식당 + {TIME_WINDOW_DAYS}일 이내 리뷰 간 연결 | {len(rwr_edges):,}개')
print('=' * 55)

: 

: 

## 다음 단계
- `04_gnn_modeling.ipynb` — GCN / GAT / GraphSAGE 모델 학습 및 성능 비교

### 설치 필요
```bash
pip install torch torch-geometric
```